# ARXIV CRAWLER

Nhận vào input là ids.txt, range id [start, end), giả sử muốn chạy từ dòng đầu tiên tới dòng 2000, thì đặt start_idx = 0, end_idx = 2000

In [48]:
IDS_TXT_DIR = "/data/arxiv-reference-matching/src/data/arxiv_id.txt"
SAVE_DIR = "/data/arxiv-reference-matching/src/data/output"
METADATA_OUTPUT_DIR = "/data/arxiv-reference-matching/src/data/metadata.json"
START_IDX = 0
END_IDX = 10

In [49]:
import arxiv
import requests
import shutil
import time
import urllib.request
from pathlib import Path

client = arxiv.Client(delay_seconds=3.0)


def get_source(arxiv_id, save_dir=SAVE_DIR):
    destination = Path(save_dir) / f"{arxiv_id}.tar.gz"
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        paper = next(client.results(arxiv.Search(id_list=[arxiv_id])))
        print(paper.title, paper.get_short_id())
        url = f"https://export.arxiv.org/src/{arxiv_id}"
        with urllib.request.urlopen(url, timeout=60) as response:
            with destination.open("wb") as output:
                shutil.copyfileobj(response, output)
        return 200
    finally:
        time.sleep(3)

In [50]:
import json
from pathlib import Path


def dump_to_json(data: list[dict]):
    if not data:
        return
    path = Path(METADATA_OUTPUT_DIR)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        with path.open("r", encoding="utf-8") as file:
            metadata = json.load(file)
    else:
        metadata = []
    metadata.extend(data)
    with path.open("w", encoding="utf-8") as file:
        json.dump(metadata, file, ensure_ascii=False, indent=2)

In [51]:
def get_id_list():
    id_list = []
    with Path(IDS_TXT_DIR).open("r", encoding="utf-8") as file:
        for index, line in enumerate(file):
            if index >= END_IDX:
                break
            if index >= START_IDX and line.strip():
                id_list.append(line.strip())
    return id_list


id_list = get_id_list()

In [52]:
count = 0
metadata_list = []
for arxiv_id in id_list:
    count += 1
    try:
        res = get_source(arxiv_id=arxiv_id, save_dir=SAVE_DIR)
        state = "success" if res == 200 else "fail"
    except (OSError, StopIteration, arxiv.ArxivError, requests.exceptions.RequestException) as error:
        print(f"Không tải được {arxiv_id}: {type(error).__name__}")
        state = "fail"
    metadata_list.append({
        "id": arxiv_id,
        "path": str(Path(SAVE_DIR) / f"{arxiv_id}.tar.gz"),
        "state": state,
    })
    if count % 100 == 0:
        dump_to_json(metadata_list)
        metadata_list.clear()

dump_to_json(metadata_list)
metadata_list.clear()

"Oh LLM, I'm Asking Thee, Please Give Me a Decision Tree": Zero-Shot Decision Tree Induction and Embedding with Large Language Models 2409.18594v1
"Oh LLM, I'm Asking Thee, Please Give Me a Decision Tree": Zero-Shot Decision Tree Induction and Embedding with Large Language Models 2409.18594v2
Dynamic Competition for Attention 2409.18595v1
Dynamic Competition for Attention 2409.18595v2
ASAG2024: A Combined Benchmark for Short Answer Grading 2409.18596v1
TemporalPaD: a reinforcement-learning framework for temporal feature representation and dimension reduction 2409.18597v1
Spectral extremal problems on outerplanar and planar graphs 2409.18598v1
Spectral extremal problems on outerplanar and planar graphs 2409.18598v2
Spectral extremal problems on outerplanar and planar graphs 2409.18598v3
Deformation maps in proto-twilled Leibniz algebras 2409.18599v1


In [ ]:
import gzip
import shutil
import tarfile
import tempfile
import zlib
from pathlib import Path

skipped_ids = []

for archive_path in sorted(Path(SAVE_DIR).glob("*.tar.gz")):
    output_dir = archive_path.parent / archive_path.name.removesuffix(".tar.gz")
    temp_path = None
    try:
        output_dir.mkdir(parents=True, exist_ok=True)
        with archive_path.open("rb") as source:
            signature = source.read(5)
        if signature.startswith(b"%PDF-"):
            print(f"Bỏ qua {archive_path.name}: chỉ có PDF, không có nguồn LaTeX")
            skipped_ids.append(output_dir.name)
            continue

        if tarfile.is_tarfile(archive_path):
            with tarfile.open(archive_path, "r:*") as archive:
                members = [
                    member for member in archive
                    if member.isfile() and Path(member.name).suffix.lower() in {".tex", ".bib"}
                ]
                # Chặn đường dẫn trong archive ghi ra ngoài thư mục đích.
                archive.extractall(output_dir, members=members, filter="data")
            kept_count = len(members)
        elif signature.startswith(b"\x1f\x8b"):
            with gzip.open(archive_path, "rb") as source:
                prefix = source.read(16384)
                if prefix.startswith(b"%PDF-"):
                    print(f"Bỏ qua {archive_path.name}: GZIP chứa PDF")
                    skipped_ids.append(output_dir.name)
                    continue
                if not any(marker in prefix for marker in (
                    b"\\documentclass", b"\\documentstyle", b"\\begin{document}", b"\\input"
                )):
                    print(f"Bỏ qua {archive_path.name}: GZIP không nhận diện được nguồn LaTeX")
                    skipped_ids.append(output_dir.name)
                    continue
                with tempfile.NamedTemporaryFile(dir=output_dir, delete=False) as output:
                    temp_path = Path(output.name)
                    output.write(prefix)
                    shutil.copyfileobj(source, output)
            temp_path.replace(output_dir / "main.tex")
            kept_count = 1
        else:
            print(f"Bỏ qua {archive_path.name}: định dạng không hỗ trợ")
            skipped_ids.append(output_dir.name)
            continue

        for file_path in output_dir.rglob("*"):
            if (file_path.is_file() or file_path.is_symlink()) and file_path.suffix.lower() not in {".tex", ".bib"}:
                file_path.unlink()
        print(f"Đã giải nén {archive_path.name}: giữ {kept_count} file .tex/.bib")
    except (tarfile.TarError, OSError, EOFError, zlib.error) as error:
        skipped_ids.append(output_dir.name)
        print(f"Không giải nén được {archive_path.name}: {error}")
    finally:
        if temp_path is not None:
            temp_path.unlink(missing_ok=True)

pass_path = Path(SAVE_DIR) / "pass.txt"
pass_path.parent.mkdir(parents=True, exist_ok=True)
pass_path.write_text("".join(f"{arxiv_id}\n" for arxiv_id in skipped_ids), encoding="utf-8")
